In [1]:
import os
%pwd
os.chdir('../')

In [2]:
import os

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/kanishka-rani-2005/WineQuality.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "kanishka-rani-2005"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "dd63b5a10fac2560fe65b7432e4c4410c6a1c7ac"

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir:Path
    test_data_path:Path
    model_path:Path
    all_params:dict
    metric_file_name:Path
    target_column:str
    mlflow_uri:str

In [4]:
from WineQuality.constants import *
from WineQuality.utils.common import read_yaml,create_directories

class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):

                 self.config=read_yaml(config_filepath)
                 self.params=read_yaml(params_filepath)
                 self.schema=read_yaml(schema_filepath)                 
                 create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self)->ModelEvaluationConfig:
            config=self.config.model_evaluation
            params=self.params.ElasticNet
            schema=self.schema.TARGET_COLUMN

            create_directories([config.root_dir])
            model_evaluation_config=ModelEvaluationConfig(
                    root_dir=config.root_dir,
                    test_data_path=config.test_data_path,
                    model_path=config.model_path,
                    all_params=params,
                    metric_file_name=config.metric_file_name,
                    target_column=schema.name,
                    mlflow_uri="https://dagshub.com/kanishka-rani-2005/WineQuality.mlflow",    
            )                 
                    
            return model_evaluation_config

In [5]:
import os 
from WineQuality import logger
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
import pandas as pd
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib
from WineQuality.utils.common import save_json

class ModelEvaluation:
    def __init__(self,config:ModelEvaluationConfig):
        self.config=config

    def eval_metrics(self,actual,pred):
        try:
            rmse=np.sqrt(mean_squared_error(actual,pred))
            mae=mean_absolute_error(actual,pred)
            r2=r2_score(actual,pred)
            return rmse,mae,r2

        except Exception as e:
            logger.error(e)
            raise e
    def log_into_mlflow(self):
        test_data=pd.read_csv(self.config.test_data_path)
        model=joblib.load(self.config.model_path)

        test_x=test_data.drop([self.config.target_column],axis=1)
        test_y=test_data[[self.config.target_column]]


        mlflow.set_registry_uri(self.config.mlflow_uri)

        tracking_url_type_store=urlparse(mlflow.get_tracking_uri()).scheme


        with mlflow.start_run():
            predicted_qualities=model.predict(test_x)
            (rmse,mae,r2)=self.eval_metrics(test_y,predicted_qualities)

            scores={"rmse":rmse,"mae":mae,"r2":r2}

            save_json(path=Path(self.config.metric_file_name),data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse",rmse)
            mlflow.log_metric("mae",mae)
            mlflow.log_metric("r2",r2)

            if tracking_url_type_store!='file':
                mlflow.sklearn.log_model(model,"model",registered_model_name='ElasticNetmodel')
            else:
                mlflow.sklearn.log_model(model,"model",registered_model_name='model')

            

e:\WineQuality\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
try:
    config=ConfigurationManager()
    model_evaluation_config=config.get_model_evaluation_config()
    model_evaluation=ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    logger.error(e)
    raise e

FILE: config\config.yaml
TYPE: <class 'dict'>
CONTENT: {'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://github.com/entbappy/Branching-tutorial/raw/master/winequality.zip', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_dir': 'artifacts/data_ingestion'}, 'data_validation': {'root_dir': 'artifacts/data_validation', 'unzip_data_dir': 'artifacts/data_ingestion/winequality-red.csv', 'STATUS_FILE': 'artifacts/data_validation/status.txt'}, 'data_transformation': {'root_dir': 'artifacts/data_transformation', 'data_path': 'artifacts/data_ingestion/winequality-red.csv'}, 'model_trainer': {'root_dir': 'artifacts/model_trainer', 'train_data_path': 'artifacts/data_transformation/train.csv', 'test_data_path': 'artifacts/data_transformation/test.csv', 'model_name': 'model.joblib'}, 'model_evaluation': {'root_dir': 'artifacts/model_evaluation', 'test_data_path': 'artifacts/data_transformation/test.csv', 'model_path': 'art

2026/06/13 11:19:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/13 11:19:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'ElasticNetmodel' already exists. Creating a new version of this model...
2026/06/13 11:19:43 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNetmodel, version 2
Created version '2' of model 'ElasticNetmodel'.


🏃 View run bold-fly-831 at: https://dagshub.com/kanishka-rani-2005/WineQuality.mlflow/#/experiments/0/runs/b82116478c1c4c67a03499b44e1890a6
🧪 View experiment at: https://dagshub.com/kanishka-rani-2005/WineQuality.mlflow/#/experiments/0
